# Task 4: Build a Speech-to-Reasoning Pipeline with Whisper & Quantized LLM

**Arch Technologies — Generative AI Internship (Month 2)**

Official flow in **one Colab notebook**:

**audio → OpenAI Whisper ASR → Unsloth dynamic 4-bit reasoning LLM → logical answer**

| Brief requirement | Where it is |
|---|---|
| Whisper transcribes audio to text | §4–6 encoding + decode |
| Fine-tuned quantized reasoner (Llama or Qwen) | §5 `Llama-3.2-3B-Instruct-unsloth-bnb-4bit` |
| Transcript used as the reasoning prompt | `reason()` / `reason_batch()` |
| Encoding | 16 kHz mono waveform → 30 s pad/trim → log-mel; tokenizer + attention mask |
| Batching | Whisper `decode` on stacked mels; LLM `generate` with left-padding |
| GPU memory | VRAM reports, `inference_mode`, cache, token caps, optional Whisper offload |
| End-to-end sample audio query | §8 featured clip |

### How to run
1. Runtime → Change runtime type → **T4 GPU**
2. Runtime → **Run all**

## 1. GPU check

In [1]:
import torch

assert torch.cuda.is_available(), (
    "No GPU detected. In Colab: Runtime → Change runtime type → T4 GPU, then Runtime → Run all."
)
print(torch.cuda.get_device_name(0), f"{torch.cuda.get_device_properties(0).total_memory/1024**3:.2f} GB")

Tesla T4 14.56 GB


## 2. Install dependencies

In [2]:
%%capture
import os, re

if "COLAB_" not in "".join(os.environ.keys()):
    !pip install -q unsloth
else:
    import torch
    v = re.match(r"[\d]{1,}\.[\d]{1,}", str(torch.__version__)).group(0)
    xformers = "xformers==" + {"2.10": "0.0.34", "2.9": "0.0.33.post1", "2.8": "0.0.32.post2"}.get(v, "0.0.34")
    !pip install -q sentencepiece protobuf "datasets==4.3.0" "huggingface_hub>=0.34.0,<1.0" hf_transfer
    !pip install -q --no-deps unsloth_zoo bitsandbytes accelerate {xformers} peft trl triton unsloth
    !pip install -q --no-deps --upgrade "torchao>=0.16.0"
    !pip install -q transformers==4.56.2
    !pip install -q --no-deps trl==0.22.2

!pip install -q openai-whisper gTTS gradio soundfile
!pip install -q --upgrade "huggingface_hub>=0.34.0,<1.0"

## 3. GPU memory plan

Whisper and the LLM both occupy VRAM. On a free T4 (~15 GB):

1. Use Whisper **`base`** (not `large`) so ASR stays small
2. Use a **3B Instruct** Unsloth dynamic 4-bit checkpoint (already SFT-tuned, then quantized)
3. Encode audio to a **30-second** Whisper window so mel tensors have a fixed shape (needed for batching)
4. Cap generation at **256** new tokens
5. Use `torch.inference_mode()` and clear the CUDA cache between stages

`Instruct` = the model was already fine-tuned for following instructions. `unsloth-bnb-4bit` = Unsloth **dynamic 4-bit** quantization (sensitive weights kept at higher precision). That matches the brief's "fine-tuned, quantized reasoning model (e.g. Unsloth dynamic 4-bit Llama or Qwen)".

Fallback if you OOM: `WHISPER_DEVICE = "cpu"` or `WHISPER_SIZE = "tiny"`, then restart the runtime.

In [3]:
import gc
import torch

def vram_report(tag: str) -> None:
    allocated = torch.cuda.memory_allocated() / 1024**3
    reserved = torch.cuda.memory_reserved() / 1024**3
    total = torch.cuda.get_device_properties(0).total_memory / 1024**3
    print(f"[{tag}] allocated={allocated:.2f} GB | reserved={reserved:.2f} GB | total={total:.2f} GB")

WHISPER_SIZE = "base"
WHISPER_DEVICE = "cuda"
MODEL_NAME = "unsloth/Llama-3.2-3B-Instruct-unsloth-bnb-4bit"
MAX_SEQ_LENGTH = 2048
MAX_NEW_TOKENS = 256
ASR_BATCH_SIZE = 2

torch.cuda.empty_cache()
gc.collect()
vram_report("start")

[start] allocated=0.00 GB | reserved=0.00 GB | total=14.56 GB


## 4. Load OpenAI Whisper

In [4]:
import whisper

asr_model = whisper.load_model(WHISPER_SIZE, device=WHISPER_DEVICE)
print("Whisper:", WHISPER_SIZE, "on", WHISPER_DEVICE)
print("mel bins:", asr_model.dims.n_mels)
vram_report("after Whisper")

100%|███████████████████████████████████████| 139M/139M [00:03<00:00, 47.4MiB/s]


Whisper: base on cuda
mel bins: 80
[after Whisper] allocated=0.27 GB | reserved=0.42 GB | total=14.56 GB


## 5. Load the quantized reasoning LLM

`FastLanguageModel.from_pretrained(..., load_in_4bit=True)` loads the Unsloth dynamic NF4 checkpoint. `for_inference` enables Unsloth's faster decode kernels. Left-padding is required for **batched** decoder-only generation.

In [5]:
from unsloth import FastLanguageModel

model, tokenizer = FastLanguageModel.from_pretrained(
    model_name=MODEL_NAME,
    max_seq_length=MAX_SEQ_LENGTH,
    dtype=None,
    load_in_4bit=True,
)
FastLanguageModel.for_inference(model)

tokenizer.padding_side = "left"
if tokenizer.pad_token_id is None:
    tokenizer.pad_token = tokenizer.eos_token

print("LLM:", MODEL_NAME)
print("pad_token_id:", tokenizer.pad_token_id, "| padding_side:", tokenizer.padding_side)
vram_report("after Unsloth LLM")

🦥 Unsloth: Will patch your computer to enable 2x faster free finetuning.
🦥 Unsloth Zoo will now patch everything to make training faster!
==((====))==  Unsloth 2026.8.19: Fast Llama patching. Transformers: 4.56.2.
   \\   /|    Tesla T4. Num GPUs = 1. Max memory: 14.563 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.11.0+cu128. CUDA: 7.5. CUDA Toolkit: 12.8. Triton: 3.6.0
\        /    Bfloat16 = FALSE. FA [Xformers = 0.0.34. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!
LLM: unsloth/Llama-3.2-3B-Instruct-unsloth-bnb-4bit
pad_token_id: 128004 | padding_side: left
[after Unsloth LLM] allocated=2.50 GB | reserved=3.35 GB | total=14.56 GB


## 6. Encoding, batching, and the pipeline

**Audio encoding (Whisper)**
- `whisper.load_audio` resamples to **16 kHz** float32 mono
- `pad_or_trim` makes a fixed **30 s** buffer so a batch of mels shares one shape
- `log_mel_spectrogram` is the encoder input Whisper actually sees

**Text encoding (LLM)**
- Chat template → tokenizer with **padding + attention_mask + truncation**
- Batched `generate` with left padding so pad tokens sit on the left

**Batching**
- ASR: stack mels and call `whisper.decode` once per mini-batch
- LLM: tokenize a list of transcripts and generate in one forward batch

In [6]:
from pathlib import Path
import torch
from gtts import gTTS

REASONING_SYSTEM = (
    "You are a careful reasoning assistant. "
    "The user message is an automatic speech transcript and may contain ASR errors. "
    "Solve the request with short step-by-step logic. "
    "End with one line: Final answer: <result>"
)


def encode_audio(audio_path: str) -> torch.Tensor:
    # 16 kHz waveform -> 30 s pad/trim -> log-mel on the ASR device
    wav = whisper.load_audio(audio_path)
    wav = whisper.pad_or_trim(wav)
    mel = whisper.log_mel_spectrogram(wav, n_mels=asr_model.dims.n_mels)
    return mel.to(asr_model.device)


def transcribe_batch(audio_paths: list[str], language: str = "en") -> list[str]:
    texts = []
    options = whisper.DecodingOptions(
        language=language,
        fp16=WHISPER_DEVICE == "cuda",
        without_timestamps=True,
    )
    for start in range(0, len(audio_paths), ASR_BATCH_SIZE):
        chunk = audio_paths[start : start + ASR_BATCH_SIZE]
        mels = torch.stack([encode_audio(p) for p in chunk])
        decoded = whisper.decode(asr_model, mels, options)
        if not isinstance(decoded, list):
            decoded = [decoded]
        texts.extend(d.text.strip() for d in decoded)
        del mels
        torch.cuda.empty_cache()
    return texts


def transcribe(audio_path: str, language: str = "en") -> str:
    return transcribe_batch([audio_path], language=language)[0]


def build_prompt(transcript: str) -> str:
    messages = [
        {"role": "system", "content": REASONING_SYSTEM},
        {
            "role": "user",
            "content": (
                "Speech transcript (from Whisper):\n"
                f"{transcript}\n\n"
                "Reason step by step, then answer."
            ),
        },
    ]
    return tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)


def reason_batch(transcripts: list[str], max_new_tokens: int = MAX_NEW_TOKENS) -> list[str]:
    prompts = [build_prompt(t) for t in transcripts]
    encoded = tokenizer(
        prompts,
        return_tensors="pt",
        padding=True,
        truncation=True,
        max_length=MAX_SEQ_LENGTH,
    ).to(model.device)
    with torch.inference_mode():
        output_ids = model.generate(
            **encoded,
            max_new_tokens=max_new_tokens,
            temperature=0.3,
            top_p=0.9,
            do_sample=True,
            use_cache=True,
            pad_token_id=tokenizer.pad_token_id,
        )
    prompt_len = encoded["input_ids"].shape[1]
    answers = tokenizer.batch_decode(output_ids[:, prompt_len:], skip_special_tokens=True)
    del encoded, output_ids
    torch.cuda.empty_cache()
    return [a.strip() for a in answers]


def reason(transcript: str, max_new_tokens: int = MAX_NEW_TOKENS) -> str:
    return reason_batch([transcript], max_new_tokens=max_new_tokens)[0]


def speech_to_reason(audio_path: str) -> dict:
    transcript = transcribe(audio_path)
    return {"transcript": transcript, "reasoning": reason(transcript)}


def speech_to_reason_batch(audio_paths: list[str]) -> list[dict]:
    transcripts = transcribe_batch(audio_paths)
    answers = reason_batch(transcripts)
    return [{"transcript": t, "reasoning": a} for t, a in zip(transcripts, answers)]


def synthesize(text: str, out_path: str) -> str:
    Path(out_path).parent.mkdir(parents=True, exist_ok=True)
    gTTS(text=text, lang="en").save(out_path)
    return out_path

print("Encoding + batched pipeline ready.")

Encoding + batched pipeline ready.


## 7. Create sample audio

gTTS writes mp3 files so the notebook is reproducible without a microphone. The featured query in §8 is the required end-to-end demo.

In [7]:
SAMPLE_QUERY = (
    "A farmer has seventeen sheep. All but nine run away. "
    "How many sheep does the farmer have left? Explain the wording, then give the number."
)

EXTRA_QUERIES = {
    "discount_tax": (
        "A laptop costs eight hundred dollars. There is a fifteen percent discount, "
        "then eight percent sales tax on the discounted price. What is the final price?"
    ),
    "compare": (
        "Which is larger, two to the power of ten or ten times twenty? Reason out loud, then pick one."
    ),
}

featured_path = synthesize(SAMPLE_QUERY, "samples/featured_query.mp3")
extra_paths = [synthesize(text, f"samples/{name}.mp3") for name, text in EXTRA_QUERIES.items()]
print("Featured audio:", featured_path)
print("Extra audio:", extra_paths)

Featured audio: samples/featured_query.mp3
Extra audio: ['samples/discount_tax.mp3', 'samples/compare.mp3']


## 8. End-to-end sample audio query

This cell is the required demonstration: one spoken question → Whisper transcript → quantized LLM reasoning.

In [8]:
print("SAMPLE AUDIO:", featured_path)
print("SPOKEN TEXT:\n", SAMPLE_QUERY)
vram_report("before featured query")

result = speech_to_reason(featured_path)

print("\nWHISPER TRANSCRIPT:\n", result["transcript"])
print("\nQUANTIZED LLM REASONING:\n", result["reasoning"])
vram_report("after featured query")

assert result["transcript"], "Whisper returned an empty transcript."
print("\nPipeline OK: audio → transcript → reasoning")

SAMPLE AUDIO: samples/featured_query.mp3
SPOKEN TEXT:
 A farmer has seventeen sheep. All but nine run away. How many sheep does the farmer have left? Explain the wording, then give the number.
[before featured query] allocated=2.50 GB | reserved=3.35 GB | total=14.56 GB

WHISPER TRANSCRIPT:
 A farmer has 17 sheep. All but nine run away. How many sheep does the farmer have left? Explain the wording. Then give the number.

QUANTIZED LLM REASONING:
 To solve this problem, let's break it down step by step:

1. The farmer has 17 sheep initially.
2. "All but nine run away" means that 9 sheep did not run away.
3. Since all the sheep except 9 ran away, the remaining sheep are the ones that did not run away.
4. Therefore, the number of sheep the farmer has left is the number of sheep that did not run away, which is 9.

Final answer: 9
[after featured query] allocated=2.61 GB | reserved=3.37 GB | total=14.56 GB

Pipeline OK: audio → transcript → reasoning


## 9. Batched extra queries

Two more clips are transcribed and answered in **mini-batches** (ASR batch size 2, then one padded LLM generate).

In [9]:
vram_report("before batch")
batch_out = speech_to_reason_batch(extra_paths)
vram_report("after batch")

for (name, original), out in zip(EXTRA_QUERIES.items(), batch_out):
    print("=" * 80)
    print("CLIP:", name)
    print("ORIGINAL:", original)
    print("TRANSCRIPT:", out["transcript"])
    print("REASONING:\n", out["reasoning"])
    print()

[before batch] allocated=2.61 GB | reserved=3.37 GB | total=14.56 GB
[after batch] allocated=2.73 GB | reserved=3.38 GB | total=14.56 GB
CLIP: discount_tax
ORIGINAL: A laptop costs eight hundred dollars. There is a fifteen percent discount, then eight percent sales tax on the discounted price. What is the final price?
TRANSCRIPT: A laptop costs $800. There is a 15% discount, then 8% sales tax on the discounted price. What is the final price?
REASONING:
 To find the final price, I'll follow these steps:

1. Calculate the discount amount: 15% of $800 = 0.15 * 800 = $120
2. Subtract the discount from the original price: $800 - $120 = $680
3. Calculate the sales tax: 8% of $680 = 0.08 * 680 = $54.40
4. Add the sales tax to the discounted price: $680 + $54.40 = $734.40

Final answer: $734.40

CLIP: compare
ORIGINAL: Which is larger, two to the power of ten or ten times twenty? Reason out loud, then pick one.
TRANSCRIPT: which is larger, 2 to the power of 10 or 10 times 20. Reason out loud, 

## 10. Optional: upload your own audio

In [10]:
from pathlib import Path

UPLOAD_AUDIO = False
uploaded = {}

if UPLOAD_AUDIO:
    from google.colab import files
    print("Upload .wav / .mp3 / .m4a")
    uploaded = files.upload()
else:
    print("Skipping upload. Set UPLOAD_AUDIO = True and re-run this cell.")

for fname, raw in uploaded.items():
    path = Path(fname).name
    Path(path).write_bytes(raw)
    out = speech_to_reason(path)
    print("=" * 80)
    print("FILE:", path)
    print("TRANSCRIPT:", out["transcript"])
    print("REASONING:\n", out["reasoning"])

Skipping upload. Set UPLOAD_AUDIO = True and re-run this cell.


## 11. Optional Gradio mic

Not required by the brief. Skip if VRAM is tight.

In [11]:
LAUNCH_GRADIO = False

if LAUNCH_GRADIO:
    import gradio as gr

    def ui_pipeline(audio):
        if audio is None:
            return "No audio received.", ""
        out = speech_to_reason(audio)
        return out["transcript"], out["reasoning"]

    gr.Interface(
        fn=ui_pipeline,
        inputs=gr.Audio(sources=["microphone", "upload"], type="filepath", label="Speak a reasoning question"),
        outputs=[
            gr.Textbox(label="Whisper transcript", lines=3),
            gr.Textbox(label="LLM reasoning", lines=14),
        ],
        title="Speech-to-Reasoning — Whisper + Unsloth 4-bit",
        description=f"ASR: Whisper {WHISPER_SIZE} · Reasoner: {MODEL_NAME}",
    ).launch(share=True)
else:
    print("Gradio off. Set LAUNCH_GRADIO = True and re-run if you want a microphone UI.")

Gradio off. Set LAUNCH_GRADIO = True and re-run if you want a microphone UI.


## 12. Checklist vs the official brief

- [x] Google Colab notebook
- [x] OpenAI Whisper ASR
- [x] Unsloth dynamic 4-bit Instruct Llama (fine-tuned + quantized; Qwen swap is one `MODEL_NAME` change)
- [x] Transcript passed as the reasoning / QA prompt
- [x] Audio encoding (16 kHz, pad/trim, log-mel) and text encoding (chat template, pad, mask)
- [x] Batching for ASR and LLM
- [x] GPU memory: 4-bit load, inference mode, cache clears, token cap, VRAM logs
- [x] Full pipeline on a sample audio query (§8)

To use Qwen instead: set `MODEL_NAME = "unsloth/Qwen2.5-3B-Instruct-unsloth-bnb-4bit"` and re-run from the model-load cell.